# MicroPhaseLab Lesson 1: From Polygons to Segmentation Masks

This notebook uses synthetic data to explain the first data pipeline. From the project root, run `microphaselab demo`, then `microphaselab baseline --manifest examples/demo/processed/manifest.csv --output-dir outputs/baseline/demo`. Launch it with `jupyter lab` from the project root, open this file, and execute each cell.

## Learning objectives

1. Distinguish SEM images, polygon annotations, and pixel masks.
2. Understand why masks contain only 0 and 1.
3. Understand why data should be split by sample group rather than random image.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

root = Path('../examples/demo')
manifest = pd.read_csv(root / 'processed/manifest.csv')
manifest

In [ ]:
row = manifest.iloc[0]
image = np.asarray(Image.open(row.image_path).convert('L'))
mask = np.asarray(Image.open(row.mask_path))
print('image shape:', image.shape)
print('mask values:', np.unique(mask))
print('MA area fraction:', mask.mean())

## Why is pixel accuracy insufficient?

If MA occupies only 5% of an image, an all-background prediction still has 95% accuracy while finding no MA at all. Dice, IoU, precision, and recall are more informative metrics.

## Read the classical baseline results

After running the `microphaselab baseline` command from the README, read the aggregate and per-image metrics. The synthetic bright regions were designed to be easy to segment, so high scores only show that the pipeline works; they do not represent performance on real steel data.

The saved prediction masks contain values of 0 and 1. A standard image viewer may render value 1 as almost black because it expects a 0–255 intensity range. The cells below display those masks with the correct scale.

In [ ]:
import json

baseline_root = Path('../outputs/baseline/demo')
summary = json.loads((baseline_root / 'summary.json').read_text(encoding='utf-8'))
metrics = pd.read_csv(baseline_root / 'metrics_per_image.csv')
summary_keys = [
    'mean_dice', 'mean_iou', 'mean_precision', 'mean_recall',
    'mean_area_fraction_absolute_error',
]
{key: summary[key] for key in summary_keys}

In [ ]:
per_image_columns = [
    'image_id', 'true_positive', 'false_positive', 'false_negative',
    'dice', 'iou', 'precision', 'recall',
]
metrics[per_image_columns]

## Inspect one prediction and its errors

The lowest-Dice image is selected automatically. White pixels are correct MA predictions, red pixels are false positives (extra predicted MA), and blue pixels are false negatives (missed MA).

In [ ]:
import matplotlib.pyplot as plt

worst_row = metrics.loc[metrics['dice'].idxmin()]
image_id = worst_row.image_id
truth_path = manifest.loc[manifest['image_id'] == image_id, 'mask_path'].iloc[0]
truth = np.asarray(Image.open(truth_path))
prediction = np.asarray(Image.open(worst_row.prediction_path))

false_positive = (prediction == 1) & (truth == 0)
false_negative = (prediction == 0) & (truth == 1)
true_positive = (prediction == 1) & (truth == 1)

comparison = np.zeros((*truth.shape, 3))
comparison[true_positive] = [1, 1, 1]
comparison[false_positive] = [1, 0, 0]
comparison[false_negative] = [0, 0.5, 1]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(prediction, cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Baseline prediction: {image_id}')
axes[1].imshow(comparison)
axes[1].set_title('White: correct; red: false positive; blue: false negative')
for axis in axes:
    axis.axis('off')
plt.show()

### Questions

1. Which image has the lowest Dice score? Inspect its prediction: is the error mainly false positives or false negatives?
2. If area-fraction error is small but IoU is low, what may have happened to the predicted boundary?
3. Why should these two synthetic-image scores not be used to choose parameters for real data?

### Suggested answers

1. `demo_a` has the lower Dice score. Its error is mainly false positives: the deterministic demo produces 24 false-positive pixels and 8 false-negative pixels, mostly near region boundaries.
2. The predicted region may have approximately the correct total area but be in the wrong location or have an incorrect shape or boundary. The amount of MA is similar, but it does not overlap well with the expert mask.
3. The synthetic images deliberately use simple brightness-based regions. Real steel micrographs have more variable contrast, noise, texture, imaging conditions, and annotation uncertainty. High synthetic scores therefore do not establish real-data performance or justify choosing real-data parameters.